# 03 - Results Analysis

This notebook analyzes QAOA performance and compares quantum solutions with classical baselines.

## Table of Contents
1. [Classical Baseline](#classical)
2. [Approximation Ratio Analysis](#ratio)
3. [QAOA Depth Comparison](#depth)
4. [Graph Cut Visualization](#visualization)
5. [Recursive QAOA](#rqaoa)

<a id='classical'></a>
## 1. Classical Baseline

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from src.graph_generator import GraphGenerator
from src.classical_solver import ClassicalSolver
from src.hamiltonian_builder import HamiltonianBuilder
from src.evaluation_metrics import EvaluationMetrics

# Generate test graph
gen = GraphGenerator(seed=42)
graph = gen.generate_d_regular_graph(n_nodes=10, degree=3, seed=42)

print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# Solve exactly using brute force
solver = ClassicalSolver(seed=42)
exact_result = solver.solve_exact(graph)

print(f"\nExact Solution:")
print(f"  Optimal cut value: {exact_result.optimal_value}")
print(f"  Optimal bitstring: {exact_result.optimal_bitstrings[0]}")
print(f"  Number of solutions: {exact_result.n_solutions}")
print(f"  Runtime: {exact_result.runtime:.4f}s")

In [ ]:
# Visualize the optimal solution
optimal_bitstring = exact_result.optimal_bitstrings[0]
partition = [i for i, b in enumerate(optimal_bitstring) if b == '0']

plt.figure(figsize=(10, 6))

pos = nx.spring_layout(graph, seed=42)

# Color nodes by partition
node_colors = ['red' if i in partition else 'blue' for i in graph.nodes()]
nx.draw(graph, pos, with_labels=True, node_color=node_colors, 
        node_size=400, edge_color='gray')

# Highlight cut edges
cut_edges = [(u, v) for u, v in graph.edges() 
              if (u in partition) != (v in partition)]
nx.draw_networkx_edges(graph, pos, edgelist=cut_edges, 
                        edge_color='green', width=3)

plt.title(f'Optimal Max-Cut Solution\n(Cut value: {exact_result.optimal_value})')
plt.axis('off')
plt.show()

print(f"Cut edges: {cut_edges}")

<a id='ratio'></a>
## 2. Approximation Ratio Analysis

In [ ]:
from src.qaoa_optimizer import MaxCutQAOAProblem, QAOAOptimizer
from src.runtime_executor import RuntimeExecutor

# Test different depths using the shared backend-aware workflow
n_qubits = graph.number_of_nodes()
depths = [1]
results = {}

for p in depths:
    print(f"\nTesting p={p}...")
    problem = MaxCutQAOAProblem(
        graph=graph,
        p=p,
        executor=RuntimeExecutor(mode='noisy_simulator', backend_name='ibm_brisbane', shots=512, seed=42, simulate_noise=True),
        seed=42,
        analysis_shots=512,
        analysis_mode='same_backend',
    )

    optimizer = QAOAOptimizer(
        p=p,
        optimizer_type='SPSA',
        maxiter=12,
        seed=42,
        n_initial_points=1,
    )
    opt_result = optimizer.optimize(
        objective_function=problem.objective_function,
        n_qubits=n_qubits,
        graph=graph,
        solution_decoder=problem.decode_solution,
    )

    results[p] = {
        'expected_cut_value': float(opt_result.cut_value),
        'sampled_cut_value': float(opt_result.sampled_cut_value) if opt_result.sampled_cut_value is not None else None,
        'objective_value': float(opt_result.optimal_value),
        'bitstring': opt_result.solution_bitstring,
        'most_likely_bitstring': opt_result.most_likely_bitstring,
        'params': opt_result.optimal_params,
    }

    print(f"  Expected cut value: {opt_result.cut_value:.4f}")
    print(f"  Representative bitstring: {opt_result.solution_bitstring}")
    print(f"  Representative sampled cut: {opt_result.sampled_cut_value}")


In [ ]:
# Compute approximation ratios from expected cut values
metrics = EvaluationMetrics()

print("Approximation Ratio Analysis:")
print("=" * 50)

ratios = []

for p in depths:
    qaoa_value = results[p]['expected_cut_value']
    ratio = metrics.compute_approximation_ratio(qaoa_value, exact_result.optimal_value)
    ratios.append(ratio)
    
    quality = metrics.assess_quality(ratio)
    
    print(f"p={p}: Expected cut={qaoa_value:.2f}, Optimal={exact_result.optimal_value}, Ratio={ratio:.4f} ({quality}), Representative bitstring={results[p]['bitstring']}, Sampled cut={results[p]['sampled_cut_value']}")

print("=" * 50)
print("Approximation ratios are computed from expected cut values, not from a decoded single bitstring.")


<a id='depth'></a>
## 3. QAOA Depth Comparison

In [ ]:
# Plot approximation ratio vs depth
plt.figure(figsize=(10, 6))

plt.plot(depths, ratios, 'o-', color='blue', linewidth=2, markersize=10)
plt.fill_between(depths, ratios, alpha=0.3)

# Reference lines
plt.axhline(y=0.8, color='green', linestyle='--', linewidth=2, label='Target (r=0.8)')
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Optimal (r=1.0)')

plt.xlabel('QAOA Depth (p)', fontsize=12)
plt.ylabel('Approximation Ratio (r)', fontsize=12)
plt.title('Approximation Ratio vs QAOA Depth', fontsize=14, fontweight='bold')
plt.xlim(0.5, 3.5)
plt.ylim(0, 1.1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<a id='visualization'></a>
## 4. Graph Cut Visualization

In [ ]:
from src.visualization import Visualizer

# Create visualizer
viz = Visualizer()

# Get cut edges
cut_edges = [(u, v) for u, v in graph.edges() 
             if (u in partition) != (v in partition)]

# Visualize optimal solution
fig = viz.plot_graph_cut(
    graph=graph,
    partition=partition,
    cut_edges=cut_edges,
    title=f'Optimal Max-Cut Solution\n(Cut value: {exact_result.optimal_value})'
)

plt.show()

<a id='rqaoa'></a>
## 5. Recursive QAOA (RQAOA)

In [ ]:
from src.rqaoa_engine import RQAOAEngine

# Optional offline RQAOA demo on a slightly larger valid graph (n * d must be even)
graph_large = gen.generate_d_regular_graph(n_nodes=8, degree=3, seed=123)

# Get exact solution for comparison
solver = ClassicalSolver(seed=42)
exact_large = solver.solve_exact(graph_large)

print(f"Larger Graph: {graph_large.number_of_nodes()} nodes")
print(f"Optimal value: {exact_large.optimal_value}")


In [ ]:
# Run RQAOA
rqaoa = RQAOAEngine(
    p=1,
    n_eliminate_per_step=1,
    correlation_threshold=0.7,
    min_problem_size=4,
    max_depth=4,
    correlation_method='counts',
    analysis_shots=512
)

# Solve
rqaoa_result = rqaoa.solve(
    graph=graph_large,
    optimal_value=exact_large.optimal_value
)

print(f"\nRQAOA Results:")
print(f"  Solution: {rqaoa_result.solution_bitstring}")
print(f"  Cut value: {rqaoa_result.cut_value}")
print(f"  Optimal value: {rqaoa_result.optimal_value}")
print(f"  Approximation ratio: {rqaoa_result.approximation_ratio:.4f}")
print(f"  Recursion levels: {rqaoa_result.n_levels}")
print(f"  Problem reduced: {rqaoa_result.original_size} → {rqaoa_result.reduced_size}")
print(f"  Runtime: {rqaoa_result.runtime:.4f}s")

## Summary

In this notebook, we covered:

1. **Classical Baseline**: Exact brute-force solution for comparison
2. **Approximation Ratio**: Computing $r = \mathbb{E}[C] / C_{\mathrm{opt}}$ from the optimized QAOA state
3. **Depth Comparison**: Comparing true multi-layer QAOA runs across $p$
4. **Graph Visualization**: Visualizing representative cut solutions
5. **Recursive QAOA**: Scaling to larger valid graph instances

### Key Findings:

- **The expected cut value is the quantity optimized by QAOA**
- **A sampled bitstring can outperform the expectation on a single instance without changing the optimized objective**
- **Approximation ratios must be interpreted per graph family and benchmark setup**
- **RQAOA** reduces problem size while preserving an equivalent objective up to a tracked constant term
